# LinkedIn Visualizations

Este notebook genera visualizaciones de los principales resultados del proyecto Olist.

Las métricas utilizadas provienen de la capa Gold y no se realizan nuevas
transformaciones sobre el pipeline.

Objetivos:

- Comparar satisfacción entre pedidos retrasados y no retrasados.
- Analizar la relación entre severidad del retraso y reviews negativas.
- Generar visualizaciones para documentación y publicación en LinkedIn.

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

In [0]:
df_satisfaction = spark.table(
    "olist.gold.customer_satisfaction_metrics"
)

df_severity = spark.table(
    "olist.gold.delay_severity_metrics"
)

In [0]:
display(df_satisfaction)
display(df_severity)

In [0]:
satisfaction = df_satisfaction.toPandas()
severity = df_severity.toPandas()

In [0]:
display(satisfaction)
display(severity)

In [0]:
satisfaction["delivery_status"] = satisfaction[
    "is_late_delivery_business"
].map({
    False: "Sin retraso",
    True: "Con retraso"
})

satisfaction = satisfaction.sort_values(
    "is_late_delivery_business"
)

In [0]:
fig, ax = plt.subplots(figsize=(9, 6))

bars = ax.bar(
    satisfaction["delivery_status"],
    satisfaction["negative_review_percentage"]
)

ax.set_title(
    "Reviews negativas según cumplimiento de entrega",
    fontsize=16,
    fontweight="bold",
    pad=18
)

ax.set_ylabel("Reviews negativas (%)", fontsize=12)
ax.set_xlabel("")

ax.set_ylim(
    0,
    satisfaction["negative_review_percentage"].max() + 15
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for bar, value in zip(
    bars,
    satisfaction["negative_review_percentage"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        f"{value:.2f}%",
        ha="center",
        fontsize=13,
        fontweight="bold"
    )

plt.tight_layout()
plt.savefig("/Workspace/Users/nicord2002@gmail.com/linkedin_delay_severity.png", dpi=300, bbox_inches="tight")
plt.show()

In [0]:
fig, ax = plt.subplots(figsize=(9, 6))

bars = ax.bar(
    satisfaction["delivery_status"],
    satisfaction["avg_review_score"]
)

ax.set_title(
    "Satisfacción promedio según cumplimiento de entrega",
    fontsize=16,
    fontweight="bold",
    pad=18
)

ax.set_ylabel(
    "Review promedio (1–5)",
    fontsize=12
)

ax.set_xlabel("")

ax.set_ylim(0, 5)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for bar, value in zip(
    bars,
    satisfaction["avg_review_score"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.12,
        f"{value:.2f}",
        ha="center",
        fontsize=13,
        fontweight="bold"
    )

plt.tight_layout()

plt.show()

In [0]:
severity_order = [
    "EARLY",
    "ON_TIME",
    "LATE_1_3",
    "LATE_4_7",
    "LATE_8_15",
    "LATE_16_PLUS"
]

severity_labels = {
    "EARLY": "Anticipado",
    "ON_TIME": "A tiempo",
    "LATE_1_3": "1–3 días",
    "LATE_4_7": "4–7 días",
    "LATE_8_15": "8–15 días",
    "LATE_16_PLUS": "+16 días"
}

severity["delay_severity"] = pd.Categorical(
    severity["delay_severity"],
    categories=severity_order,
    ordered=True
)

severity = severity.sort_values("delay_severity")

severity["delay_label"] = (
    severity["delay_severity"]
    .astype(str)
    .map(severity_labels)
)

display(severity)

In [0]:
fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.bar(
    severity["delay_label"],
    severity["negative_review_percentage"]
)

ax.set_title(
    "Severidad del retraso y reviews negativas",
    fontsize=16,
    fontweight="bold",
    pad=18
)

ax.set_ylabel(
    "Reviews negativas (%)",
    fontsize=12
)

ax.set_xlabel(
    "Condición de entrega",
    fontsize=12
)

ax.set_ylim(
    0,
    severity["negative_review_percentage"].max() + 12
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for bar, value in zip(
    bars,
    severity["negative_review_percentage"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.5,
        f"{value:.1f}%",
        ha="center",
        fontsize=11,
        fontweight="bold"
    )

plt.tight_layout()
plt.savefig("/Workspace/Users/nicord2002@gmail.com/linkedin_negative_reviews.png", dpi=300, bbox_inches="tight")
plt.show()